In [0]:
import pandas as pd

# List of worksheets in your Google Spreadsheet
sheet_names = [
    "Students", 
    "Batches", 
    "Student_Batch", 
    "Classes", 
    "Platforms", 
    "Student_Access", 
    "Assignments", 
    "Attendance", 
    "Assignment_Submissions"
]

SPREADSHEET_ID = "1_KqnfO8FgnSJpNLezha8yQRMgBqJjg864nVql7tD_ME"
spark.sql("CREATE CATALOG rivadataplatform")
spark.sql("CREATE SCHEMA landing")
TARGET_DATABASE = "rivadataplatform.landing"  

for sheet in sheet_names:
    # Build Google's public CSV export endpoint
    url = f"https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/gviz/tq?tqx=out:csv&sheet={sheet}"
 
    print(f"Fetching sheet '{sheet}'...")
    df_pd = pd.read_csv(url)
    
    # Sanitize column names for Delta Lake compatibility
    df_pd.columns = [col.strip().replace(" ", "_").lower() for col in df_pd.columns]
    
    # Convert empty strings/NaN to string types safely for PySpark
    df_spark = spark.createDataFrame(df_pd)
    
    table_name = sheet.strip().replace(" ", "_").lower()
    
    # Save as Delta Table in Databricks
    df_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f"{TARGET_DATABASE}.{table_name}")
    
    print(f"Successfully created table: {TARGET_DATABASE}.{table_name}")

print("All worksheets loaded successfully!")

In [0]:
url